# Laboratorio 2 · Bitácora

**Nombre:**  
**Usuario de GitHub:**  
**Fecha:**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla:** la predicción se escribe ANTES de ejecutar la celda de código
> que tiene debajo. Equivocarse no resta. Rellenarla después, sí.

> **Lo nuevo de hoy:** los métodos de esta sesión son aleatorios. Una sola
> ejecución no es una medición. A partir del ejercicio 2, todo número que
> escribas aquí tiene que venir con su intervalo y con cuántas semillas lo
> produjeron.


## Preparación


In [ ]:
import numpy as np

from rlrs.dp import value_iteration
from rlrs.envs import ARROWS, GridWorld, acantilado
from rlrs.evaluation import evaluate
from rlrs.policies import EpsilonAvidaPolicy, GreedyTabularPolicy
from rlrs.td import error_frente_a, mc_control, q_learning, sarsa

# Si esta celda falla, para y resuélvelo antes de seguir.
print('todo importado')


## Mi variante

La misma de ayer. Si no la anotaste, ejecuta `uv run python scripts/variante.py`.


In [ ]:
RUIDO = None   # <- rellena, el mismo de ayer
COSTE = None   # <- rellena, el mismo de ayer
GAMMA = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)

# La respuesta conocida: tu V* de ayer. Es contra esto que medimos hoy.
optimos, politica_optima, barridos = value_iteration(mi_env, gamma=GAMMA)
print(f'{barridos} barridos'); print(mi_env.render_values(optimos, politica_optima))


### Dos ayudas que se usan en todo el cuaderno


In [ ]:
libres = [(r, c) for r in range(mi_env.n_rows) for c in range(mi_env.n_cols)
          if not mi_env.is_wall((r, c)) and not mi_env.is_terminal((r, c))]


def coincidencias(q):
    """En cuántas casillas la acción ávida de q es la acción óptima."""
    return sum(int(q[mi_env.state_index(p)].argmax()
                   == politica_optima[mi_env.state_index(p)]) for p in libres)


def intervalo(xs):
    """Media e intervalo de confianza al 95 %. Devuelve (media, bajo, alto)."""
    a = np.array(xs, dtype=float)
    media = a.mean()
    mitad = 1.96 * a.std(ddof=1) / np.sqrt(len(a)) if len(a) > 1 else 0.0
    return media, media - mitad, media + mitad


print(f'{len(libres)} casillas libres')


---
## Ejercicio 1 · Los tres métodos contra la respuesta conocida


**Antes de ejecutar.** Ordena los tres métodos de menor a mayor error, y di por qué crees que ese es el orden.

_Tu predicción:_  



In [ ]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=0)
    err = error_frente_a(ap.q, optimos, mi_env)
    print(f'{nombre:<12} error {err:.4f}   política {coincidencias(ap.q)}/{len(libres)}')


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  



---
## Ejercicio 2 · Un número sin intervalo, otra vez

> Esta celda tarda cerca de medio minuto. No se colgó.


**Antes de ejecutar.** ¿Se va a mantener el orden del ejercicio 1 con cinco semillas? ¿Y van las dos cifras, el error y el recuento de política, a contar la misma historia?

_Tu predicción:_  



In [ ]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    errores, politicas = [], []
    for semilla in range(5):
        ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=semilla)
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'{nombre:<12} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


**Con los intervalos delante, responde las dos por separado.**

1. ¿El **error** distingue a los tres métodos, o hay parejas cuyos intervalos se solapan?

   _Tu respuesta:_  

2. ¿El **recuento de política** los distingue?

   _Tu respuesta:_  



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  



---
## Ejercicio 3 · Apagar la exploración


**Antes de ejecutar.** Con $\varepsilon = 0$ el agente siempre toma la acción que ahora mismo cree mejor. ¿Aprenderá la política óptima, una peor, o depende de la suerte inicial? Y con $\varepsilon = 0{,}5$: ¿mejor o peor que con $0{,}1$?

_Tu predicción:_  



In [ ]:
for eps in (0.0, 0.05, 0.1, 0.3, 0.5):
    errores, politicas = [], []
    for semilla in range(5):
        ap = sarsa(mi_env, episodes=5000, gamma=GAMMA,
                   epsilon=eps, epsilon_final=eps, seed=semilla)   # sin decaimiento
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'eps {eps:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


**Son dos fallos distintos.** Nombra por separado qué le falta al agente de $\varepsilon = 0$ y qué le sobra al de $\varepsilon = 0{,}5$.

_Tu respuesta:_  



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  



---
## Ejercicio 4 · El tamaño del paso


**Antes de ejecutar.** ¿El error va a bajar monótonamente al subir $\alpha$, va a subir monótonamente, o va a tener un mínimo en algún punto intermedio? Apuesta por una de las tres formas.

_Tu predicción:_  



In [ ]:
for alpha in (0.01, 0.1, 0.5, 0.9):
    errores = [error_frente_a(sarsa(mi_env, episodes=5000, gamma=GAMMA,
                                    alpha=alpha, seed=s).q, optimos, mi_env)
               for s in range(5)]
    e, elo, ehi = intervalo(errores)
    print(f'alpha {alpha:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]')


**Distingue los dos problemas.** El de $\alpha$ muy pequeño y el de $\alpha$ muy grande no son el mismo.

_Tu respuesta:_  



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  



---
## Ejercicio 5 · El error plantado

Este no lleva código propio. Ejecuta en la terminal:

```
uv run python experiments/sin_modelo.py --parte 3
```


**Antes de ejecutar.** ¿Cuál de las dos formas de medir va a dar un retorno peor, y por qué? ¿Y cuánto peor, un poco o mucho?

_Tu predicción:_  



**Pega aquí la salida del guion.**

```

```



**El diagnóstico.** ¿Por qué esa medición está mal hecha, y qué está midiendo en realidad? Y en una frase: ¿cuándo sí tendría sentido medir con la política que explora?

_Tu respuesta:_  



---
## Ejercicio 6 · El acantilado


**Antes de ejecutar.** ¿Cuál de los dos métodos va a ganar? Escríbelo, y después vuelve a leer la pregunta: ¿tiene sentido tal como está formulada?

_Tu predicción:_  



In [ ]:
cl = acantilado()

for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    avido, explorando, entrenamiento = [], [], []
    for semilla in range(5):
        ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=semilla)
        ev = evaluate(acantilado(), GreedyTabularPolicy(ap.q.argmax(axis=1)),
                      episodes=100, base_seed=0)
        avido.append(ev.mean)
        ex = evaluate(acantilado(), EpsilonAvidaPolicy(ap.q, 0.05),
                      episodes=100, base_seed=0)
        explorando.append(ex.mean)
        entrenamiento.append(float(np.mean(ap.retornos[-500:])))
    a, alo, ahi = intervalo(avido)
    x, xlo, xhi = intervalo(explorando)
    t, tlo, thi = intervalo(entrenamiento)
    print(f'{nombre:<11} ávido {a:+.2f} [{alo:+.2f}, {ahi:+.2f}]'
          f'   explorando {x:+.2f} [{xlo:+.2f}, {xhi:+.2f}]'
          f'   entrenamiento {t:+.2f} [{tlo:+.2f}, {thi:+.2f}]')


### Los dos caminos


In [ ]:
for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=0)
    print(f'\n{nombre}:')
    print(cl.render_values(ap.q.max(axis=1), ap.q.argmax(axis=1)))


**La explicación del mecanismo.** Escribe las dos reglas de actualización una debajo de la otra y subraya lo único que cambia: qué valor se usa para el estado siguiente. Desde ahí, explica por qué cada método aprende el camino que aprende.

_Tu respuesta:_  



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  



---
## Antes de entregar

- [ ] Las seis predicciones están escritas, y se escribieron antes de ejecutar.
- [ ] Todos los números llevan su intervalo y dicen cuántas semillas los produjeron.
- [ ] Ninguna conclusión dice más de lo que los intervalos permiten decir.
- [ ] Las explicaciones de los ejercicios 5 y 6 hablan del mecanismo, no del resultado.
- [ ] **Kernel → Restart & Run All**, y el cuaderno corre entero de arriba abajo.
- [ ] `git add`, `git commit -m "Laboratorio 2"`, `git push`.
- [ ] Las dos líneas pegadas en Moodle.
